# 02. 수동 스킬 적용 경로 (사람이 통제)

`01`이 만든 레거시 데이터셋(`artifacts/demo/dataset.csv`)을 바탕으로, 이 저장소의 스킬을 사람이 통제하는 기본 경로로 실행해 개선합니다.

```text
legacy-intake -> AIDM experiment -> AIDD promotion -> human review
```

이 노트북은 외부 도구가 아니라 **이 저장소의 스킬 러너**(`.agents/scripts/run-aidm.sh`, `.agents/scripts/verify-promotion.sh`)를 직접 호출합니다. 각 러너는 `aidm-experiment`·`aidd-promotion` 스킬이 감싸는 실행체와 동일합니다. 데모를 보는 사람은 자기 레거시(`01`)를 코딩 에이전트에게 맡겼을 때, 하네스가 **제안을 검증하고 게이트로 보호하며 사람 검토 지점에서 멈추는** 방식을 확인할 수 있습니다.

- 먼저 `01_legacy_baseline.ipynb`를 실행해 데이터셋을 만들어 두세요.
- 에이전트는 코드가 아니라 선언형 JSON 제안만 제출합니다.
- 승격 게이트(`decision`)를 우회하지 않으며, AIDD는 검증·생성이지 배포가 아닙니다.
- 우승 후보뿐 아니라 평가된 모든 대체 후보와 SPOT 기준선의 순위를 함께 표시합니다.

In [6]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def repo_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("power-forecasting 저장소 안에서 실행하세요.")


REPO_ROOT = repo_root(Path.cwd())
SCRIPTS = REPO_ROOT / ".agents" / "scripts"
CATALOG = REPO_ROOT / "configs" / "optimization-catalog.v1.json"


def runner_env():
    # GUI로 띄운 Jupyter 커널은 uv/venv가 PATH에 없을 수 있어, 러너가 찾도록 보강합니다.
    env = dict(os.environ)
    extra = [str(Path(sys.executable).parent)]
    uv = shutil.which("uv") or str(Path.home() / ".local" / "bin" / "uv")
    if Path(uv).exists():
        extra.append(str(Path(uv).parent))
    env["PATH"] = os.pathsep.join(extra + [env.get("PATH", "")])
    return env


RUNNER_ENV = runner_env()
DEMO = REPO_ROOT / "artifacts" / "demo"
DATASET = DEMO / "dataset.csv"
assert DATASET.exists(), "먼저 01_legacy_baseline.ipynb 를 실행해 artifacts/demo/dataset.csv 를 만들어 주세요."

RUN_DIR = REPO_ROOT / "runs" / "notebook-02-manual"
PROPOSAL = REPO_ROOT / "runs" / "notebook-02-proposal.json"
shutil.rmtree(RUN_DIR, ignore_errors=True)
PROPOSAL.parent.mkdir(parents=True, exist_ok=True)

# aidm-experiment 스킬: 에이전트는 코드가 아니라 선언형 JSON 제안만 제출합니다.
# 예측 시점 입력(달력, 예보 물리량)만 사용하고 generation_mw / actual_* 는 쓰지 않습니다.
proposal = {
    "schema_version": "1",
    "proposal_id": "manual-demo-candidates",
    "rationale": "Compare bounded prediction-time candidates against the SPOT baseline.",
    "baseline": {"model": "SPOT"},
    "feature_sets": [
        {
            "name": "forecast_only",
            "rationale": "Forecast-time cloud-adjusted irradiance only.",
            "specs": [
                {"name": "effective_irradiance", "transform": "effective_irradiance", "inputs": ["forecast_irradiance", "forecast_cloud_cover"], "parameters": {}, "rationale": "Cloud-adjusted forecast irradiance.", "version": "1"},
            ],
        },
        {
            "name": "calendar_physics",
            "rationale": "Forecast-time solar physics plus daily calendar signals.",
            "specs": [
                {"name": "hour_sin", "transform": "cyclic_hour", "inputs": ["timestamp"], "parameters": {}, "rationale": "Daily cycle sine.", "version": "1"},
                {"name": "hour_cos", "transform": "cyclic_hour", "inputs": ["timestamp"], "parameters": {}, "rationale": "Daily cycle cosine.", "version": "1"},
                {"name": "effective_irradiance", "transform": "effective_irradiance", "inputs": ["forecast_irradiance", "forecast_cloud_cover"], "parameters": {}, "rationale": "Cloud-adjusted forecast irradiance.", "version": "1"},
            ],
        },
    ],
    "model_recipes": [
        {"name": "ridge_demo", "recipe": "ridge", "parameters": {"alpha": 1.0}, "rationale": "Regularized linear recipe for comparison."},
        {"name": "hgb_demo", "recipe": "hist_gradient_boosting", "parameters": {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 31}, "rationale": "Bounded deterministic boosted trees."},
    ],
    "budget": {"max_evaluations": 4, "top_feature_groups": 2},
}
PROPOSAL.write_text(json.dumps(proposal, ensure_ascii=False, indent=2), encoding="utf-8")

# 이 저장소의 aidm-experiment 스킬 러너를 01의 데이터셋에 그대로 실행합니다.
# 운영 기본 게이트(최소 개선율 0.01, 발전소별 최대 저하 0.03)는 러너 기본값을 사용합니다.
aidm = subprocess.run(
    [str(SCRIPTS / "run-aidm.sh"), "--dataset", str(DATASET), "--catalog", str(CATALOG), "--proposal", str(PROPOSAL), "--run-dir", str(RUN_DIR), "--folds", "5"],
    cwd=REPO_ROOT, text=True, capture_output=True, env=RUNNER_ENV,
)
if aidm.returncode != 0:
    raise RuntimeError(f"run-aidm.sh 실패 (exit {aidm.returncode}):\n{aidm.stderr}")

manifest = json.loads((RUN_DIR / "promotion_manifest.json").read_text(encoding="utf-8"))
print(f"결정(decision): {manifest['decision']}")
print(f"SPOT 기준선 NMAE: {manifest['baseline']['metrics']['nmae']:.6f}")
print(f"우승 후보: {manifest['winner']['name']}  NMAE: {manifest['winner']['metrics']['nmae']:.6f}")
print(f"개선율(improvement_ratio): {manifest['improvement_ratio']:.6f}")
print(f"실패한 게이트: {manifest['failed_gates']}")
print(f"평가한 후보 수(기준선 제외): {len(proposal['feature_sets']) * len(proposal['model_recipes'])}")

결정(decision): promote
SPOT 기준선 NMAE: 0.015922
우승 후보: hgb_demo:calendar_physics  NMAE: 0.014422
개선율(improvement_ratio): 0.094199
실패한 게이트: []
평가한 후보 수(기준선 제외): 4


In [ ]:
import sqlite3

import pandas as pd

# aidm-experiment 증적: experiments.db 에는 우승뿐 아니라 모든 후보와 SPOT 기준선이 기록됩니다.
con = sqlite3.connect(RUN_DIR / "experiments.db")
try:
    rows = con.execute("select name, status, metrics_json from runs").fetchall()
finally:
    con.close()

baseline_nmae = manifest["baseline"]["metrics"]["nmae"]
records = []
for name, status, metrics_json in rows:
    metrics = json.loads(metrics_json) if metrics_json else {}
    nmae = metrics.get("nmae")
    if nmae is None:
        continue
    if name.startswith("aidm-baseline"):
        label, kind = "SPOT (baseline)", "baseline"
    elif name.startswith("aidm-proposal-"):
        label, kind = name[len("aidm-proposal-"):], "candidate"
    else:
        continue
    records.append({
        "후보": label,
        "구분": kind,
        "상태": status,
        "NMAE": nmae,
        "기준선대비개선율": (baseline_nmae - nmae) / baseline_nmae,
    })

ranking = pd.DataFrame(records).sort_values("NMAE").reset_index(drop=True)
winner_name = manifest["winner"]["name"]
ranking["선정"] = ["★ 우승" if row == winner_name else "" for row in ranking["후보"]]
print("우승뿐 아니라 모든 후보와 SPOT 기준선을 NMAE 오름차순으로 표시합니다 (낮을수록 좋음).")
ranking

In [7]:
if manifest["decision"] == "promote":
    # aidd-promotion 스킬 러너: 승격 매니페스트만 검증·컴파일하고 사람 검토용 요청을 만듭니다.
    verify = subprocess.run(
        [str(SCRIPTS / "verify-promotion.sh"), "--run-dir", str(RUN_DIR)],
        cwd=REPO_ROOT, text=True, capture_output=True, env=RUNNER_ENV,
    )
    if verify.returncode != 0:
        raise RuntimeError(f"verify-promotion.sh 실패 (exit {verify.returncode}):\n{verify.stderr}")
    evidence = json.loads((RUN_DIR / "promotion-evidence.json").read_text(encoding="utf-8"))
    module = RUN_DIR / "generated" / "promoted_features.py"
    print(f"검증 상태: {evidence['status']}")
    print(f"생성된 모듈: {module}")
    print("--- 앞부분 미리보기 ---")
    print("\n".join(module.read_text(encoding="utf-8").splitlines()[:12]))
    patch = RUN_DIR / "model-recipe-patch.json"
    if patch.exists():
        print(f"\n사람 검토용 모델 레시피 요청: {json.loads(patch.read_text(encoding='utf-8'))['status']}")
    print("\n[사람 검토 경계] 생성 모듈·매니페스트·patch는 검토 대상이며, 고객 시스템 배포는 사람 승인 이후에만 진행합니다.")
else:
    print("게이트가 승격을 거부했습니다. 하네스가 개선이 아닌 변경을 막았다는 뜻입니다.")
    print(f"실패한 게이트: {manifest['failed_gates']}")
    print("이 경우 AIDD를 실행하지 않으며, 제안을 수정해 다시 실험합니다.")

검증 상태: success
생성된 모듈: /Users/andy/works/ai/ml-harness/runs/notebook-02-manual/generated/promoted_features.py
--- 앞부분 미리보기 ---
# This file is generated by power_forecasting.aidd; do not edit.
# Manifest schema_version: '1'
# Manifest winner: 'effective_irradiance+hour_cos+hour_sin'
from __future__ import annotations

import pandas as pd

from power_forecasting.features import FeatureSpec, apply_feature_specs


MANIFEST_SCHEMA_VERSION = '1'
MANIFEST_WINNER_NAME = 'effective_irradiance+hour_cos+hour_sin'

사람 검토용 모델 레시피 요청: requires_human_review

[사람 검토 경계] 생성 모듈·매니페스트·patch는 검토 대상이며, 고객 시스템 배포는 사람 승인 이후에만 진행합니다.


## 단계와 스킬 대응

| 단계 | 스킬 / 러너 | 반드시 멈추는 경계 |
| --- | --- | --- |
| 레거시 연결·기준선 | `legacy-intake` (01의 데이터셋·SPOT 예측) | 사람 승인 전에는 실제 고객 시스템·데이터를 실행하지 않음 |
| 후보 탐색·게이트 비교 | `aidm-experiment` (`run-aidm.sh`) | `decision: reject`를 우회하지 않고 AIDD를 호출하지 않음 |
| 매니페스트 검증·모듈 생성 | `aidd-promotion` (`verify-promotion.sh`) | 고객 저장소 수정·병합·배포를 하지 않음 |
| 종합 판정 | `release-gate` | 명시적 사람 승인 없이 release를 허용하지 않음 |

이 경로의 가치는 자동 개선 자체가 아니라, **개선을 주장하기 전에 검증 가능한 증적과 게이트를 남긴다는 점**입니다. 실행 증적은 `runs/notebook-02-manual/`의 `promotion_manifest.json`, `generated/promoted_features.py`, `promotion-evidence.json`, `model-recipe-patch.json`입니다.